# GRPO pilot: Qwen2.5-3B-Instruct + QLoRA on MedQA

The point isn't the results. It's to measure **memory, time per step, and whether training is stable**, then estimate how many GPU hours the full run needs.

Run in order: check GPU -> set variables -> pull the repo -> install dependencies -> mount Drive -> (optional) HF_TOKEN -> unit tests -> data sample -> pilot run -> report and plots -> GPU-hour estimate -> copy results to Drive

Before running: **Runtime -> Change runtime type -> GPU** (T4 or A100).

In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 needs Python >= 3.11'

In [ ]:
# 2. Set variables (this is the only cell you need to edit)
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'  # <- your repo
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'

OUTPUT_DIR = '/content/outputs/grpo_pilot_bf16_vllm'                    # local Colab output folder (lost on disconnect); the folder name shows up in the result folder name on Drive
DRIVE_DIR  = '/content/drive/MyDrive/medqa-calibration/grpo_pilot'      # where results get saved on Google Drive

TRL_VERSION = '1.16.0'   # the code is written for this version; check the "parameters to check" note before changing it

# pilot settings (defaults match train/grpo/grpo.py)
N_QUESTIONS           = 64
SEED                  = 42
MAX_STEPS             = 30
NUM_GENERATIONS       = 4
MAX_COMPLETION_LENGTH = 512
LEARNING_RATE         = 1e-5
BETA                  = 0.04   # KL coefficient; 0 = skip KL (faster)
TEMPERATURE           = 1.0
PER_DEVICE_BS         = 4      # answers per forward/backward (if you run out of memory, lower this and raise GRAD_ACCUM)
GRAD_ACCUM            = 4      # PER_DEVICE_BS x GRAD_ACCUM has to be a multiple of NUM_GENERATIONS
LORA_R, LORA_ALPHA    = 16, 32
LORA_TARGET_MODULES   = 'all-linear'
USE_VLLM              = True   # only recommended on an A100
QUANTIZATION          = 'none'  # '4bit' = QLoRA; 'none' = bf16 + LoRA (use 'none' with vLLM: QLoRA + vLLM makes the gradients fade out)

assert (PER_DEVICE_BS * GRAD_ACCUM) % NUM_GENERATIONS == 0, 'PER_DEVICE_BS x GRAD_ACCUM has to be divisible by NUM_GENERATIONS'
print('per step:', PER_DEVICE_BS * GRAD_ACCUM, 'answers =', PER_DEVICE_BS * GRAD_ACCUM // NUM_GENERATIONS, 'questions x', NUM_GENERATIONS, 'answers')

In [ ]:
# 3. git clone / pull repo
# Private repo: add GITHUB_TOKEN (a GitHub personal access token) under the key icon (Secrets) on the left in Colab, and turn on "Notebook access"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # no GITHUB_TOKEN set, treat it as a public repo

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'train/grpo/test_rewards.py', 'train/grpo/grpo.py', 'train/grpo/estimate.py']:
    assert Path(f).exists(), f'{f} not found in the repo: make sure it has been pushed to {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. Install dependencies
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM installs the torch version it wants, so you usually need Runtime -> Restart session afterwards, then rerun from the "set variables" cell
    # vLLM 0.28+ dropped bitsandbytes quantization, but with QLoRA (4-bit) TRL has vLLM load the model through bitsandbytes
    # -> pinned to 0.27.1 (the last version with bitsandbytes, inside TRL 1.16's allowed 0.22-0.31 range, needs torch==2.13.0)
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab's preinstalled torchao 0.10 is too old: peft 0.21 errors out when adding LoRA without quantization (--quantization none)
# "only versions above 0.16.0 are supported". We don't use torchao (neither vllm nor peft needs it), so just uninstall it
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} not installed')
print('\nIf pip says a restart is needed above (or imports fail later): Runtime -> Restart session, then rerun from the "set variables" cell.')

In [ ]:
# 5. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive folder:', DRIVE_DIR)

In [ ]:
# 6. (Optional) set HF_TOKEN: Qwen2.5 and MedQA are public so downloads work without it, but with it they're more reliable and less likely to get rate-limited
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN set')
except Exception:
    print('No HF_TOKEN found (optional), downloading anonymously')

In [ ]:
# 7. Run the reward-function unit tests (a few seconds, no GPU)
!python -m pytest -q train/grpo/test_rewards.py

In [ ]:
# 8. Look at the data: one raw example + the converted prompt (check the fields, and that the prompt matches eval/)
!python -m train.grpo.grpo_data --n {N_QUESTIONS}

In [ ]:
# 9. Pilot run (about 30-60 min on a T4, 10-20 min on an A100; results get copied to DRIVE_DIR at the end)
args = [
    f'--n_questions {N_QUESTIONS}', f'--seed {SEED}', f'--max_steps {MAX_STEPS}',
    f'--num_generations {NUM_GENERATIONS}', f'--max_completion_length {MAX_COMPLETION_LENGTH}',
    f'--learning_rate {LEARNING_RATE}', f'--beta {BETA}', f'--temperature {TEMPERATURE}',
    f'--per_device_train_batch_size {PER_DEVICE_BS}', f'--gradient_accumulation_steps {GRAD_ACCUM}',
    f'--lora_r {LORA_R}', f'--lora_alpha {LORA_ALPHA}', f'--lora_target_modules {LORA_TARGET_MODULES}',
    f'--output_dir {OUTPUT_DIR}', f'--drive_dir {DRIVE_DIR}',
]
if USE_VLLM:
    args.append('--use_vllm')
args.append(f'--quantization {QUANTIZATION}')
ARGS = ' '.join(args)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('python -m train.grpo.grpo', ARGS)
!python -u -m train.grpo.grpo {ARGS} 2>&1 | tee {OUTPUT_DIR}/train_log.txt

In [ ]:
# 10. Report and plots
import json
import pandas as pd
from IPython.display import Image, display

report = json.load(open(f'{OUTPUT_DIR}/pilot_report.json'))
print('GPU:', report['gpu_name'], '| TRL:', report['trl_version'], '| precision:', report['environment']['precision'])
print('avg time per step (s, warmup excluded):', report['timing']['avg_step_time_s_excl_warmup'])
print('peak memory allocated / reserved / total (GB):', report['memory']['peak_mem_allocated_gb_max_over_steps'],
      report['memory']['peak_mem_reserved_gb_max_over_steps'], report['memory']['gpu_total_memory_gb'])
print('reward:', report['reward'])
print('format rate:', report['format_rate'])
print('truncation rate (mean):', report['truncated_frac_mean'])
print('all-same group share (mean / last steps):', report['frac_groups_all_same_mean'], report['frac_groups_all_same_last_k_mean'])
print('stability:', report['stability'])
print('config TRL actually used:', json.dumps(report['effective_grpo_config'], ensure_ascii=False))
if report.get('dropped_config_keys'):
    print('Heads up, ignored by this TRL version:', report['dropped_config_keys'])

df = pd.read_csv(f'{OUTPUT_DIR}/metrics.csv')
cols = ['step', 'step_time_s', 'peak_mem_allocated_gb', 'reward_mean', 'reward_std', 'frac_groups_all_same',
        'format_rate', 'mean_completion_tokens', 'truncated_frac', 'kl', 'grad_norm']
display(df[[c for c in cols if c in df.columns]].round(4))
display(Image(f'{OUTPUT_DIR}/curves.png'))

In [ ]:
# 11. Estimate GPU hours for the full run (from the measured time per step)
!python -m train.grpo.estimate --report {OUTPUT_DIR}/pilot_report.json --num_questions 10178 --epochs 1 --num_generations {NUM_GENERATIONS} --per_device_train_batch_size {PER_DEVICE_BS} --gradient_accumulation_steps {GRAD_ACCUM} --grid

# you can also try different setups directly in Python:
from train.grpo.estimate import estimate_from_report, format_estimate
for nq, ep in [(10178, 1), (5000, 1), (10178, 2)]:
    r = estimate_from_report(f'{OUTPUT_DIR}/pilot_report.json', num_questions=nq, epochs=ep,
                             num_generations=NUM_GENERATIONS, per_device_train_batch_size=PER_DEVICE_BS,
                             gradient_accumulation_steps=GRAD_ACCUM)
    print(format_estimate(r))

In [ ]:
# 12. Copy results to Drive (train.grpo.grpo already copied once at the end; this copies again with the full train_log.txt)
import shutil
dest = Path(DRIVE_DIR) / report['run_name']
shutil.copytree(OUTPUT_DIR, dest, dirs_exist_ok=True)
print('saved to', dest)
for p in sorted(dest.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(dest)}  ({p.stat().st_size / 1e6:.1f} MB)')